# Best-estimate Parameters Corner Plot 

In this notebook, we reproduce the corner plots in Figures 6 and 13 in Ronchi et al. (2026) for the trained density estimator from the following experiments:

- When the entire observed X-ray population is considered (for Figure 13), we use the trained posterior estimator saved on the PIC server at the following path: `/data/magnesia/common/paper_ronchi_etal_2025/B_double_lognorm_dip-tor_heavy/tsnpe_experiment_1_maps8_res32/learning/models/SBI_ConvolutionMDN/20260115_142235/round_4`.
- When only the sample of young magnetars and XDINSs is considered for inference (for Figure 6), we use the trained posterior estimator saved on the PIC server at the following path: `/data/magnesia/common/paper_ronchi_etal_2025/B_double_lognorm_dip-tor_heavy/tsnpe_experiment_1_maps8_res32_youngxdins/learning/models/SBI_ConvolutionMDN/20260202_122351/round_4`.

We also compute the Pearson and Spearman correlation coefficients, along with their corresponding p-values, for the 2D marginalized posterior distribution. Note that in order to make this notebook work, you first need to download the results data from `/data/magnesia/common/paper_ronchi_etal_2025/experiments_paper.zip` unpack the file and copy the entire folder experiments into `MAGNESIA_population_synthesis/data/paper_results/ronchi_etal_2026/experiments`.

In [ ]:
import torch
import corner
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import json

from matplotlib import rc
from scipy.stats import pearsonr, spearmanr
from IPython.display import display, HTML

#rc("text", usetex=True)
rc("font", family="serif")
#plt.rcParams["text.latex.preamble"] = r"\usepackage{amsmath}"

In [ ]:
SMALL_SIZE = 14
MEDIUM_SIZE = 20
BIGGER_SIZE = 30

plt.rc("font", size=SMALL_SIZE)  # controls default text sizes
plt.rc("axes", titlesize=MEDIUM_SIZE)  # fontsize of the axes title
plt.rc("axes", labelsize=MEDIUM_SIZE)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("ytick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("legend", fontsize=SMALL_SIZE)  # legend fontsize
plt.rc("figure", titlesize=BIGGER_SIZE)  # fontsize of the figure title

### Plotting the posterior distribution.

In [ ]:
def import_statistics(stats_path: str):
    """
    Extracting the mean and standard deviation for all the parameters in the `stats_path` file.
    
    Args:
        stats_path (str): Path to the file where the statistics are saved.
        
    Returns:
        (torch.tensor, torch.tensor): Mean and standard deviation for the parameters in the `stats_path` file.
    """
    std_list = []
    mean_list = []
    max_list = []
    min_list = []
    
    with open(stats_path, "r") as json_file:
        data = json.load(json_file)
        
    for key, value in data.items():
        std_list.append(value["std"])
        mean_list.append(value["mean"])
        max_list.append(value["max"])
        min_list.append(value["min"])
        
    mean = np.array(mean_list)
    std = np.array(std_list)
    max_list = np.array(max_list)
    min_list = np.array(min_list)
    
    return mean, std, max_list, min_list

In [ ]:
# By default, we consider the posterior inferred using the entire X-ray sample to reproduce Figure 12.
# To consider the inference results using only young magnetars and XDINSs and reproduce Figure 6, 
# we set `use_young_xdins_only` to True.

use_young_xdins_only = True

if use_young_xdins_only:
    stats_path = "../../data/paper_results/ronchi_etal_2026/experiments/tsnpe_experiment_1_maps8_res32_youngxdins/data/statistics_train.json"
    directory_path = "../../data/paper_results/ronchi_etal_2026/experiments/tsnpe_experiment_1_maps8_res32_youngxdins/learning/models/SBI_ConvolutionMDN/20260202_122351"
else:
    stats_path = "../../data/paper_results/ronchi_etal_2026/experiments/tsnpe_experiment_1_maps8_res32/data/statistics_train.json"
    directory_path = "../../data/paper_results/ronchi_etal_2026/experiments/tsnpe_experiment_1_maps8_res32/learning/models/SBI_ConvolutionMDN/20260115_142235"

observed_posterior_round = []
n_rounds = 10
for i in range(n_rounds):
    observed_posterior_round.append(torch.load(f"{directory_path}/round_{i}/samples_posterior.pt").detach().cpu().numpy())


In [ ]:
mean, std, par_max, par_min = import_statistics(stats_path)

In [ ]:
parameter_labels = [
    r"$\mu_{\log P}$",
    r"$\sigma_{\log P}$",
    r"$\mu_{\log B,1}$",
    r"$\sigma_{\log B,1}$",
    r"$\mu_{\log B,2}$",
    r"$\sigma_{\log B,2}$",
    r"$w_{\log B}$",
    r"$a_{\rm late}$",
    r"$\mu_{\log L_0}$",
    r"$\alpha_L$"

]

n_param = np.shape(observed_posterior_round)[2]

In [ ]:
# We take the posterior distribution from round 5, i.e., with index 4.
round_corner = 4
observed_samples = observed_posterior_round[round_corner]

In [ ]:
observed_samples = observed_samples * std[0:n_param] + mean[0:n_param]

quantile = np.quantile(observed_samples, [0.025, 0.5, 0.975], axis=0)

range_param = [[par_min[v], par_max[v]] for v in range(n_param)]

param_median = quantile[1, :]

figure = corner.corner(
    observed_samples,
    bins=32,
    labels=parameter_labels,
    label_kwargs={"fontsize": 22},
    #range=range_param,
    color='k',
    quantiles=[0.025, 0.5, 0.975],
    levels=(
        1 - np.exp(-0.5),
        1 - np.exp(-2),
        1 - np.exp(-9.0 / 2.0),
    ),  # 1, 2 and 3 sigma levels
    show_titles=True,
    title_kwargs={"fontsize": 18},
    rasterize = True
)

corner.overplot_lines(figure, param_median, color='tab:blue')

corner.overplot_points(
    figure,
    param_median[None],
    marker="o",
    color='tab:blue',
)

if use_young_xdins_only:
    plt.savefig('plots/corner_plot_round4_youngxdins.png',dpi=400)
else:
    plt.savefig('plots/corner_plot_round4_full.png',dpi=400)


plt.show()

### Compute the median and its correspoding 95% C.I.

In [ ]:
quantile = np.quantile(observed_samples, [0.025, 0.5, 0.975], axis=0)

medians = quantile[1, :]

deviation_lower = medians - quantile[0, :]
deviation_upper = quantile[2, :] - medians

for i, label in enumerate(parameter_labels):
    print(f"{label}: {medians[i]:.2f} -{deviation_lower[i]:.2f} +{deviation_upper[i]:.2f}")


### Compute the median and its correspoding 1-$\sigma$ C.I.

In [ ]:
quantile = np.quantile(observed_samples, [0.16, 0.5, 0.84], axis=0)
medians = quantile[1, :]

deviation_lower = medians - quantile[0, :]
deviation_upper = quantile[2, :] - medians

for i, label in enumerate(parameter_labels):
    print(f"{label}: {medians[i]:.2f} -{deviation_lower[i]:.2f} +{deviation_upper[i]:.2f}")


### Computing the Pearson and Spearman correlation coefficients
We compute Pearson and Spearman correlation coefficients, which measure the strength and direction of linear relationships and the monotonicity of the relationship between two datasets, respectively. In these statistical tests, the null hypothesis is that the random variables are not correlated. Therefore, a small p-value (e.g., p < 0.05) indicates strong evidence against the null hypothesis, suggesting that if a correlation is present, it is statistically significant. In other words, a p-value of 0.05 means there is a 5% chance of obtaining this correlation due to statistical fluctuations. We consider parameters to be correlated if and only if the absolute value of the coefficient is greater than $0.5$ and the corresponding p-value $<0.05$.

In [ ]:
n_params = observed_samples.shape[1]
pearson_matrix = np.zeros((n_params, n_params))
spearmanr_matrix = np.zeros((n_params, n_params))
p_value_matrix_pearson = np.zeros((n_params, n_params))
p_value_matrix_spearmanr = np.zeros((n_params, n_params))

for i in range(n_params):
    for j in range(n_params):
        if i == j:
            # The correlation of a variable with itself is 1, and p-value is 0.
            pearson_matrix[i, j] = 1.0
            spearmanr_matrix[i,j] = 1.0
            p_value_matrix_pearson[i, j] = 0.0
            p_value_matrix_spearmanr[i, j] = 0.0
        else:
            # Calculate Pearson correlation and p-value for pairs.
            corr_pearson, p_value_pearson = pearsonr(observed_samples[:, i], observed_samples[:, j])
            corr_spearmanr, p_value_spearmanr = spearmanr(observed_samples[:, i], observed_samples[:, j])
            pearson_matrix[i, j] = corr_pearson
            spearmanr_matrix[i,j]= corr_spearmanr
            p_value_matrix_pearson[i, j] = p_value_pearson
            p_value_matrix_spearmanr[i, j] = p_value_spearmanr

# Round the matrices to 2 decimal places.
pearson_matrix = np.round(pearson_matrix, 2)
p_value_matrix_pearson = np.round(p_value_matrix_pearson, 2)
spearmanr_matrix = np.round(spearmanr_matrix, 2)
p_value_matrix_spearmanr = np.round(p_value_matrix_spearmanr, 2)

# Convert matrices to DataFrames for better visualization with LaTeX-compatible labels.
param_names = [f"${label}$" for label in parameter_labels]  
pearson_df = pd.DataFrame(pearson_matrix, index=param_names, columns=param_names)
spearmanr_df = pd.DataFrame(spearmanr_matrix, index=param_names, columns=param_names)
p_value_pearson_df = pd.DataFrame(p_value_matrix_pearson, index=param_names, columns=param_names)
p_value_spearmanr_df = pd.DataFrame(p_value_matrix_spearmanr, index=param_names, columns=param_names)

# Display the matrices as formatted HTML tables.
print("Pearson Correlation Coefficients Matrix:")
display(HTML(pearson_df.to_html(escape=False)))

print("\nP-values Matrix for the Pearson correlation:")
display(HTML(p_value_pearson_df.to_html(escape=False)))

# Display the matrices as formatted HTML tables.
print("Spearman Correlation Coefficients Matrix:")
display(HTML(spearmanr_df.to_html(escape=False)))

print("\nP-values Matrix for the Spearman correlation:")
display(HTML(p_value_spearmanr_df.to_html(escape=False)))